# fine-tuning using your in-house assay

This tutorial is for users who have already run a functional assay and would like to answer the question "what if I had run my assay with other drugs, other CRISPR treatments, or in other cell lines?"

For example, [Tieu et al (2024)](https://doi.org/10.1016/j.cell.2024.01.035) run a combinatorial CAR-T transduction with 24 guides for a total of 576 pairwise combinations. We can fine-tune Prophet on this dataset and make predictions for genes spanning the entire genome and additional combinations therein.

In [34]:
import pandas as pd
import yaml
from prophet import Prophet
from prophet.data import universal_processing
from prophet.core.config import set_config
from prophet.utils import validate_prophet_inputs

We load in a config file to get the file paths for the embeddings which should be used. These can be downloaded from the links on Github.

In [35]:
with open("config_file_finetuning.yaml", "r") as f:
    config = set_config(yaml.safe_load(f))

We've randomly select one of the pretrained model checkpoints (noted in the config file above) to train on here. For more robust results, we recommend training with several checkpoints and taking the ensemble prediction.

In [36]:
config.ckpt_path

'/lustre/groups/ml01/projects/super_rad_project/pretrained_prophet/GDSC/cl_0_TrainedOn545_cl_out_300cl_1219iv_512model_8layers_Falsesimpler_Truemask_0.0001lr_Falseexplicitphenotype_5000warmup_40000max_iters_Falseunbalanced_0.01wd_2048bs_Trueft/cl_0_TrainedOn545_seed_1995/epoch=27-step=2688.ckpt'

Load in the pretrained model.

In [37]:
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=config.ckpt_path,
)

Learning rate set to 1e-05


Here we've provided two examples for finetuning to demonstrate the variety of datasets on which you can finetune Prophet. We recommend that all datasets have values minmaxed to between 0 and 1 before finetuning.

### GDSC2

GDSC2 (https://www.cancerrxgene.org/) is a cancer-screening dataset with titrated IC50s.

 - cell states: cancer cell lines
 - interventions: small molecule singletons
 - readout: IC50 of viability as measured using CellTitreGlo at 72hrs, fitted over multiple concentrations

In [38]:
gdsc_data_path = (
    "/lustre/groups/ml01/projects/super_rad_project/data/GDSC_notscaled_minmax.csv"
)
data_label = pd.read_csv(gdsc_data_path, index_col=0)

data_label["iv2"] = (
    "negative_drug"  # there is no second compound so we specify negative_drug so the token is masked
)
data_label["phenotype"] = (
    "GDSC"  # this is the label to use for this readout when you run inference later
)

validation_results = validate_prophet_inputs(
    df=data_label,
    iv_emb_path=model.iv_emb_path,
    cl_emb_path=model.cl_emb_path,
    ph_emb_path=model.ph_emb_path,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    readout_col="value",
    mode="train",
)

source_df = validation_results["processed_inputs"]["df"]
source_df

Starting input validation...
Validating embedding files...
Processing DataFrame with 242036 rows and 6 columns...
Validating DataFrame structure and columns...
Checking for missing values...
Converting text to lowercase...


/ictstr01/home/icb/ahmet.kaya/prophet_latest/prophet/prophet/utils/validation.py:50: UserWarning: Unexpected columns found: ['iv_name']. These will be ignored during processing.
  warnings.warn(


Loading embedding files...
Filtering data to match available embeddings...
Filtering data to match available embeddings...
Filtered 101935 rows with missing embeddings (140101 rows remaining)
Input validation completed successfully
Filtered 101935 rows with missing embeddings (140101 rows remaining)
Input validation completed successfully


,cell_line,iv1,value,iv_name,phenotype,iv2
0,PFSK1,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.323078,camptothecin,gdsc,negative_drug
1,A673,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.172422,camptothecin,gdsc,negative_drug
2,SKES1,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.160326,camptothecin,gdsc,negative_drug
3,COLO829,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.333331,camptothecin,gdsc,negative_drug
4,5637,cc[c@@]1(o)c(=o)occ2c1cc1-c3nc4ccccc4cc3cn1c2=o,0.271361,camptothecin,gdsc,negative_drug
...,...,...,...,...,...,...
140096,MM1S,cc(=o)nc(cs)c(=o)o,0.799827,n-acetyl cysteine,gdsc,negative_drug
140097,SNU175,cc(=o)nc(cs)c(=o)o,0.835833,n-acetyl cysteine,gdsc,negative_drug
140098,SNU407,cc(=o)nc(cs)c(=o)o,0.766903,n-acetyl cysteine,gdsc,negative_drug
140099,SNU61,cc(=o)nc(cs)c(=o)o,0.852908,n-acetyl cysteine,gdsc,negative_drug


In [39]:

config.dirpath = "./ckpts_GDSC/"  # set the save directory here; you can also set it in the config file
model.train(
    source_df,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    model_config=config,
     
)

Fitting model.


Concatenating gene embeddings: 1operation [00:00, 7049.25operation/s]
Concatenating cell line embeddings: 1operation [00:00, 10485.76operation/s]
Concatenating gene embeddings: 1operation [00:00, 7049.25operation/s]
Concatenating cell line embeddings: 1operation [00:00, 10485.76operation/s]
Concatenating phenotype embeddings: 1operation [00:00, 8160.12operation/s]

/home/icb/ahmet.kaya/miniconda3/envs/prophet-new-latest/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/ahmet.kaya/miniconda3/envs/prophet-new-lat ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/icb/ahmet.kaya/miniconda3/envs/prophet-new-latest/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `s

Using unbalanced sampling: False
R2 average:  False
R2 average: False
Dataset sizes:
  Training:    126,090 samples
  Validation:  14,011 samples
  Test:        0 samples


/home/icb/ahmet.kaya/miniconda3/envs/prophet-new-latest/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
/home/icb/ahmet.kaya/miniconda3/envs/prophet-new-latest/lib/python3.10/site-packages/pytorch_lightning/callbacks/model_checkpoint.py:751: Checkpoint directory /ictstr01/home/icb/ahmet.kaya/prophet_latest/prophet/tutorials/ckpts_GDSC exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name                | Type               | Params | Mode 
-------------------------------------------------------------------
0 | learnable_embedding | Embedding          | 512 K  | train
1 | embedding_dropout   | Dropout            | 0      | train
2 | gene_net            | Sequential         | 887 K  | train
3 | drug_net            | Sequential         | 887 K  | train
4 | cl_

Epoch 2: 100%|██████████| 16/16 [00:00<00:00, 17.64it/s, v_num=mj9h, train_loss=0.00202]  

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 2: 100%|██████████| 16/16 [00:00<00:00, 17.56it/s, v_num=mj9h, train_loss=0.00202]



### TieuQi

This is a T-cell proliferation dataset from https://doi.org/10.1016/j.cell.2024.01.035.

- cell state: T-cell proliferation
- intervention: combinatorial CRISPRi
- readout: Log2FC of CD8+ T-cells vs. control pDNA

In [40]:
tieu_data_path = "/ictstr01/home/icb/yuge.ji/projects/super_rad_project/data/TieuQi_lfc_day11_minmax.csv"
data_label = pd.read_csv(tieu_data_path, index_col=0)

data_label["phenotype"] = (
    "T-cell_viability"  # this is the label to use for this readout when you run inference later
)

validation_results = validate_prophet_inputs(
    df=data_label,
    iv_emb_path=model.iv_emb_path,
    cl_emb_path=model.cl_emb_path,
    ph_emb_path=model.ph_emb_path,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    readout_col="value",
    mode="train",
)

source_df = validation_results["processed_inputs"]["df"]
source_df

Starting input validation...
Validating embedding files...
Processing DataFrame with 625 rows and 5 columns...
Validating DataFrame structure and columns...
Checking for missing values...
Converting text to lowercase...
Loading embedding files...
Filtering data to match available embeddings...
Filtered 184 rows with missing embeddings (441 rows remaining)
Input validation completed successfully
Filtering data to match available embeddings...
Filtered 184 rows with missing embeddings (441 rows remaining)
Input validation completed successfully


,iv1,iv2,value,cell_line,phenotype
0,batf3,batf3,0.633443,JURKAT,t-cell_viability
1,batf3,cblb,0.544929,JURKAT,t-cell_viability
2,batf3,ctla4,0.483662,JURKAT,t-cell_viability
3,batf3,dhx37,0.693173,JURKAT,t-cell_viability
4,batf3,fas,0.594995,JURKAT,t-cell_viability
...,...,...,...,...,...
436,zc3h12a,rasa2,0.458578,JURKAT,t-cell_viability
437,zc3h12a,socs1,0.545633,JURKAT,t-cell_viability
438,zc3h12a,tox,0.596647,JURKAT,t-cell_viability
439,zc3h12a,tox2,0.418890,JURKAT,t-cell_viability


In [41]:
config.dirpath = "./ckpts_TieuQi/"
model.train(
    source_df,
    iv_col=["iv1", "iv2"],
    cl_col="cell_line",
    ph_col="phenotype",
    model_config=config,
)

Fitting model.


Concatenating gene embeddings: 1operation [00:00, 8738.13operation/s]
Concatenating cell line embeddings: 1operation [00:00, 9177.91operation/s]
Concatenating gene embeddings: 1operation [00:00, 8738.13operation/s]on/s]
Concatenating cell line embeddings: 1operation [00:00, 9177.91operation/s]
Concatenating phenotype embeddings: 1operation [00:00, 9404.27operation/s]

/home/icb/ahmet.kaya/miniconda3/envs/prophet-new-latest/lib/python3.10/site-packages/lightning_fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/icb/ahmet.kaya/miniconda3/envs/prophet-new-lat ...
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/icb/ahmet.kaya/miniconda3/envs/prophet-new-latest/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wand

Using unbalanced sampling: False
R2 average:  False
R2 average: False
Dataset sizes:
  Training:    396 samples
  Validation:  45 samples
  Test:        0 samples
Epoch 999: 100%|██████████| 1/1 [00:00<00:00,  8.90it/s, v_num=mj9h, train_loss=0.0161]

`Trainer.fit` stopped: `max_steps=1000` reached.


Epoch 999: 100%|██████████| 1/1 [00:00<00:00,  8.63it/s, v_num=mj9h, train_loss=0.0161]



## Loading in a model checkpoint

In [43]:
#replace this with your checkpoint
pretrained_checkpoint_path = "./ckpts_TieuQi/epoch=997-step=998.ckpt" 
model = Prophet(
    iv_emb_path=config.genes_prior,
    cl_emb_path=config.cell_lines_prior,
    ph_emb_path=None,
    model_pth=pretrained_checkpoint_path,
)

Learning rate set to 1e-05


As you can see, this checkpoint can now be loaded for inference! See other notebooks in `tutorials` for how to perform inference, or copy this notebook for more finetuning.